# Sandbox — Landing to Bronze

Ambiente de exploração usado para desenhar o script genérico `landing_to_bronze.py`.

**Objetivo da camada Bronze:** ler os CSVs crus da Landing Zone e persistir em Parquet, **preservando o dado original**, adicionando apenas um metadado técnico (timestamp de processamento). Nenhuma limpeza acontece aqui — isso é responsabilidade da Silver.

Aqui testamos: (1) leitura de um CSV simples, (2) o caso problemático do `reviews` (texto com quebras de linha), (3) o padrão de escrita em Parquet.

In [ ]:
from utils.spark_utils import create_spark_session
from pyspark.sql.functions import current_timestamp

spark = create_spark_session('Sandbox-Landing-to-Bronze')

## 1. Leitura de um CSV simples (customers)

Caso feliz: o CSV não tem armadilhas. Lemos com header e separador padrão.

In [ ]:
input_customers = 's3a://landing-zone/vendas/*/olist_customers_dataset.csv'

df = spark.read.option('header', 'true').option('sep', ',').csv(input_customers)
df.show(5, truncate=False)
df.printSchema()

Repare que **todas as colunas vêm como `string`** — o Spark não infere tipos na leitura de CSV por padrão, e nem queremos isso na Bronze. A tipagem correta será feita na Silver.

## 2. O caso problemático: `reviews`

A tabela de avaliações tem comentários de texto livre (`review_comment_message`) com **quebras de linha e aspas dentro do próprio campo**. Lendo de forma ingênua, o Spark quebra um comentário em vários registros inválidos.

In [ ]:
input_reviews = 's3a://landing-zone/vendas/*/olist_order_reviews_dataset.csv'

# Leitura ingênua (sem tratar texto multilinha)
df_ruim = spark.read.option('header', 'true').option('sep', ',').csv(input_reviews)
print('Linhas (leitura ingenua):', df_ruim.count())
df_ruim.printSchema()

A contagem acima fica inflada e algumas linhas aparecem com colunas deslocadas (o texto "vazou" para o registro seguinte). A correção é dizer ao Spark que um campo pode ter várias linhas e usar aspas como delimitador de texto:

In [ ]:
# Leitura correta: multiLine + quote + escape
df_bom = (spark.read
    .option('header', 'true')
    .option('sep', ',')
    .option('multiLine', 'true')
    .option('quote', '"')
    .option('escape', '"')
    .csv(input_reviews))

print('Linhas (leitura correta):', df_bom.count())
df_bom.show(5, truncate=True)

**Conclusão:** por isso o `reviews` tem `opcoes_csv` no `config/tabelas_olist.py`. O script `landing_to_bronze.py` aplica essas opções extras dinamicamente, sem hardcode — as outras tabelas usam a leitura padrão.

## 3. Padrão de escrita da Bronze

Adicionamos o timestamp de processamento e gravamos em Parquet (formato colunar, comprimido, com schema embutido — muito mais eficiente que CSV para as leituras da Silver).

In [ ]:
df_bronze = df.withColumn('data_processamento_bronze', current_timestamp())
df_bronze.show(5, truncate=False)

# No script de producao isto vira:
# df_bronze.write.mode('overwrite').parquet('s3a://bronze/olist/customers/')

In [ ]:
spark.stop()